# Pipeline automático — Idoneidad climática *Albizia guachapele*

Este notebook ejecuta el pipeline completo de punta a punta:

1. **Bronze** → consolida todos los CSV de la carpeta `presencias/` y lee el raster `bio9_HISTORICO.tif`
2. **Silver** → extrae `bio_9` en presencias, genera background de grilla, arma el dataset
3. **Gold** → partición train/test estratificada
4. **Entrenamiento** → compara modelos y selecciona el GLM cuadrático
5. **Registro** → nueva versión del modelo en Unity Catalog
6. **Serving** → actualiza el endpoint a la versión recién registrada

Se dispara automáticamente cuando llega un archivo nuevo a la carpeta
`bronze/presencias/`. Para agregar registros, sube otro CSV con las mismas
columnas (`species`, `Latitude`, `Longitude`) — el pipeline consolida todos.

In [0]:
%pip install rasterio

In [0]:
dbutils.library.restartPython()

In [0]:
# ── CONFIGURACIÓN ───────────────────────────────────────────────────
CATALOG = "especies_nativas"
SCHEMA_BRONZE = "especies_nativas"
VOLUME = "bronze"

BASE_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/{VOLUME}"
CARPETA_PRESENCIAS = f"{BASE_VOLUME}/presencias"
RASTER_HISTORICO = f"{BASE_VOLUME}/bio9_HISTORICO.tif"

TABLA_SILVER = f"{CATALOG}.silver.dataset_bio9"
TABLA_GOLD = f"{CATALOG}.gold.dataset_bio9_final"
MODELO_UC = f"{CATALOG}.models.albizia_guachapele_bio9"

ENDPOINT_NAME = "albizia-guachapele-bio9"

BUFFER_DEG = 0.0083333  # ~1 píxel (~1 km) alrededor de cada presencia
RANDOM_STATE = 42

AUC_MINIMO = 0.70

## 1. Bronze — lectura y validación de datos crudos

In [0]:
import os
import glob
import pandas as pd
import numpy as np
import rasterio

archivos = sorted(glob.glob(os.path.join(CARPETA_PRESENCIAS, "*.csv")))
if not archivos:
    raise FileNotFoundError(f"No se encontró ningún CSV en {CARPETA_PRESENCIAS}")

print(f"Archivos encontrados ({len(archivos)}):")
columnas_requeridas = {"species", "Latitude", "Longitude"}
partes = []
for ruta in archivos:
    df = pd.read_csv(ruta)
    faltantes = columnas_requeridas - set(df.columns)
    if faltantes:
        raise ValueError(f"'{os.path.basename(ruta)}' no tiene las columnas requeridas: {faltantes}")
    df["_origen"] = os.path.basename(ruta)
    partes.append(df)
    print(f"  - {os.path.basename(ruta)}: {len(df)} registros")

presencias = pd.concat(partes, ignore_index=True)
print(f"Total consolidado: {len(presencias)} registros de presencia")

antes = len(presencias)
presencias = presencias.dropna(subset=["Latitude", "Longitude"]).drop_duplicates(
    subset=["Latitude", "Longitude"]
)
print(f"Registros tras limpiar nulos/duplicados: {len(presencias)} (se removieron {antes - len(presencias)})")

In [0]:
with rasterio.open(RASTER_HISTORICO) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Tamaño:", src.width, "x", src.height)
    bounds = src.bounds

dentro = (
    (presencias["Longitude"] >= bounds.left)
    & (presencias["Longitude"] <= bounds.right)
    & (presencias["Latitude"] >= bounds.bottom)
    & (presencias["Latitude"] <= bounds.top)
)
fuera = (~dentro).sum()
if fuera:
    print(f"ADVERTENCIA: {fuera} presencias fuera del raster fueron descartadas")
presencias = presencias[dentro].reset_index(drop=True)
print(f"Presencias válidas dentro del área de estudio: {len(presencias)}")

## 2. Silver — extracción de bio_9 y generación de background

In [0]:
with rasterio.open(RASTER_HISTORICO) as src:
    coords = list(zip(presencias["Longitude"], presencias["Latitude"]))
    presencias["bio_9"] = [v[0] for v in src.sample(coords)]
    nodata = src.nodata

    arr = src.read(1)
    transform = src.transform
    rows, cols = np.where(arr != nodata)
    xs, ys = rasterio.transform.xy(transform, rows, cols)
    xs, ys = np.array(xs), np.array(ys)
    vals = arr[rows, cols]

presencias = presencias[presencias["bio_9"] != nodata].reset_index(drop=True)
print(f"Presencias con bio_9 válido: {len(presencias)}")
print(f"Rango de bio_9 en presencias: {presencias['bio_9'].min():.2f} a {presencias['bio_9'].max():.2f}")

In [0]:
mask_keep = np.ones(len(xs), dtype=bool)
for plon, plat in zip(presencias["Longitude"].values, presencias["Latitude"].values):
    dist = np.sqrt((xs - plon) ** 2 + (ys - plat) ** 2)
    mask_keep &= dist > BUFFER_DEG

background = pd.DataFrame(
    {
        "Latitude": ys[mask_keep],
        "Longitude": xs[mask_keep],
        "bio_9": vals[mask_keep],
        "presencia": 0,
    }
)

presencias_final = presencias[["Latitude", "Longitude", "bio_9"]].copy()
presencias_final["presencia"] = 1

dataset_silver = pd.concat([presencias_final, background], ignore_index=True)
print(dataset_silver["presencia"].value_counts())

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.createDataFrame(dataset_silver).write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(TABLA_SILVER)
print(f"Tabla Silver actualizada: {TABLA_SILVER}")

## 3. Gold — partición train/test estratificada

In [0]:
from sklearn.model_selection import train_test_split

dataset_gold = dataset_silver.copy()
train_idx, test_idx = train_test_split(
    dataset_gold.index,
    test_size=0.2,
    stratify=dataset_gold["presencia"],
    random_state=RANDOM_STATE,
)
dataset_gold["split"] = "train"
dataset_gold.loc[test_idx, "split"] = "test"

print(dataset_gold.groupby(["split", "presencia"]).size())

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
spark.createDataFrame(dataset_gold).write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(TABLA_GOLD)
print(f"Tabla Gold actualizada: {TABLA_GOLD}")

## 4. Entrenamiento y comparación de modelos

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

X_train = dataset_gold.loc[dataset_gold["split"] == "train", ["bio_9"]].values
y_train = dataset_gold.loc[dataset_gold["split"] == "train", "presencia"].values
X_test = dataset_gold.loc[dataset_gold["split"] == "test", ["bio_9"]].values
y_test = dataset_gold.loc[dataset_gold["split"] == "test", "presencia"].values

def glm_cuadratico():
    return make_pipeline(
        PolynomialFeatures(degree=2, include_bias=False),
        LogisticRegression(class_weight="balanced"),
    )

candidatos = {
    "GLM cuadratico": glm_cuadratico(),
    "Random Forest (regularizado)": RandomForestClassifier(
        n_estimators=100, max_depth=4, min_samples_leaf=10,
        class_weight="balanced", random_state=RANDOM_STATE,
    ),
    "Gradient Boosting (regularizado)": GradientBoostingClassifier(
        n_estimators=50, max_depth=2, learning_rate=0.05,
        min_samples_leaf=15, random_state=RANDOM_STATE,
    ),
}

resultados = []
for nombre, modelo in candidatos.items():
    cv_auc = cross_val_score(modelo, X_train, y_train, cv=5, scoring="roc_auc").mean()
    modelo.fit(X_train, y_train)
    test_auc = roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])
    resultados.append({"modelo": nombre, "cv_auc": cv_auc, "test_auc": test_auc})

tabla = pd.DataFrame(resultados).sort_values("test_auc", ascending=False)
print(tabla.to_string(index=False))

### Selección del modelo

Se mantiene el **GLM cuadrático** como modelo de producción por decisión de diseño:
aunque los modelos de árboles suelen obtener mayor AUC, producen curvas de respuesta
ecológicamente implausibles (picos artificiales) con este volumen de datos y una sola
variable. La tabla de comparación se registra en MLflow como evidencia.

In [0]:
auc_glm = tabla.loc[tabla["modelo"] == "GLM cuadratico", "test_auc"].iloc[0]
cv_glm = tabla.loc[tabla["modelo"] == "GLM cuadratico", "cv_auc"].iloc[0]

if auc_glm < AUC_MINIMO:
    raise ValueError(
        f"El AUC del modelo ({auc_glm:.4f}) está por debajo del mínimo aceptable "
        f"({AUC_MINIMO}). No se promueve el modelo a producción."
    )
print(f"Control de calidad superado: AUC test = {auc_glm:.4f} (mínimo {AUC_MINIMO})")

## 5. Registro de la nueva versión en Unity Catalog

In [0]:
import mlflow
import mlflow.pyfunc
from mlflow.models.signature import infer_signature

mlflow.set_registry_uri("databricks-uc")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.models")

X_full = np.vstack([X_train, X_test])
y_full = np.concatenate([y_train, y_test])

modelo_final = glm_cuadratico()
modelo_final.fit(X_full, y_full)
print(f"Modelo final reentrenado con {len(X_full)} registros")


class ProbabilityWrapper(mlflow.pyfunc.PythonModel):
    """Expone predict_proba() en vez de predict(), para servir el índice continuo."""

    def __init__(self, sklearn_model):
        self.sklearn_model = sklearn_model

    def predict(self, context, model_input):
        return self.sklearn_model.predict_proba(model_input)[:, 1]


signature = infer_signature(X_full, modelo_final.predict_proba(X_full)[:, 1])

with mlflow.start_run(run_name="pipeline_automatico_albizia"):
    mlflow.log_param("modelo", "GLM cuadratico (bio_9 + bio_9^2)")
    mlflow.log_param("n_presencias", int(dataset_gold["presencia"].sum()))
    mlflow.log_param("n_background", int((dataset_gold["presencia"] == 0).sum()))
    mlflow.log_metric("cv_auc_train", float(cv_glm))
    mlflow.log_metric("test_auc", float(auc_glm))
    mlflow.log_text(tabla.to_string(index=False), "comparacion_modelos.txt")

    info = mlflow.pyfunc.log_model(
        python_model=ProbabilityWrapper(modelo_final),
        artifact_path="model",
        signature=signature,
        input_example=X_full[:5],
        registered_model_name=MODELO_UC,
    )

print("Modelo registrado correctamente")

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")
versiones = client.search_model_versions(f"name='{MODELO_UC}'")
nueva_version = max(int(v.version) for v in versiones)
print(f"Nueva versión registrada: {nueva_version}")

## 6. Actualización del Serving Endpoint

In [0]:
import requests

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
HOST = ctx.apiUrl().get()
TOKEN = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

payload = {
    "served_entities": [
        {
            "entity_name": MODELO_UC,
            "entity_version": str(nueva_version),
            "workload_size": "Small",
            "scale_to_zero_enabled": True,
        }
    ]
}

resp = requests.put(
    f"{HOST}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config",
    headers=headers,
    json=payload,
    timeout=60,
)

if resp.status_code == 200:
    print(f"Endpoint '{ENDPOINT_NAME}' actualizándose a la versión {nueva_version}")
    print("El despliegue tarda unos minutos; la app usará el modelo nuevo automáticamente.")
else:
    raise RuntimeError(f"Error actualizando el endpoint ({resp.status_code}): {resp.text}")

## Resumen de la ejecución

In [0]:
print(f"""
PIPELINE COMPLETADO
-------------------
Presencias procesadas : {int(dataset_gold['presencia'].sum())}
Background generado   : {int((dataset_gold['presencia'] == 0).sum())}
AUC (validación cruz.): {cv_glm:.4f}
AUC (test)            : {auc_glm:.4f}
Versión del modelo    : {nueva_version}
Endpoint actualizado  : {ENDPOINT_NAME}
""")